In [1]:
import os

BASE_PATH = "/kaggle/working/flood_app"

os.makedirs(f"{BASE_PATH}/templates", exist_ok=True)
os.makedirs(f"{BASE_PATH}/static", exist_ok=True)

print("✅ Project folders created")

✅ Project folders created


In [2]:
html = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Flood Prediction System</title>
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
  <link rel="preconnect" href="https://fonts.googleapis.com"/>
  <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&display=swap" rel="stylesheet"/>
  <link rel="stylesheet" href="/static/style.css"/>
</head>
<body>

<!-- Animated background particles -->
<div class="bg-particles">
  <div class="particle"></div><div class="particle"></div>
  <div class="particle"></div><div class="particle"></div>
  <div class="particle"></div><div class="particle"></div>
</div>

<!-- Rain overlay -->
<div class="rain-container" id="rainContainer"></div>

<div class="container">

  <!-- Header -->
  <div class="header">
    <div class="header-badge">AI POWERED</div>
    <div class="header-icon-wrap">
      <div class="header-icon-ring"></div>
      <div class="header-icon">🌊</div>
    </div>
    <h1>Flood Prediction System</h1>
    <p>Enter an area name to get an AI-powered 3-day flood risk forecast</p>
    <div class="header-stats">
      <div class="stat"><span class="stat-dot green"></span>Model Active</div>
      <div class="stat"><span class="stat-dot blue"></span>Live Weather</div>
      <div class="stat"><span class="stat-dot purple"></span>Real Terrain</div>
    </div>
  </div>

  <!-- Search Bar -->
  <div class="search-section">
    <div class="search-wrap" id="searchWrap">
      <span class="search-icon">📍</span>
      <div class="search-input-wrap">
        <input
          type="text"
          id="areaInput"
          placeholder="Search any location... e.g. Chennai, Mumbai, Kerala"
          autocomplete="off"
        />
        <!-- Suggestions Dropdown -->
        <div id="suggestions" class="suggestions hidden"></div>
      </div>
      <button id="searchBtn" class="btn-search">
        <span class="btn-text">Search</span>
        <span class="btn-icon">→</span>
      </button>
    </div>
    <div id="searchError" class="error-msg hidden"></div>
  </div>

  <!-- Map -->
  <div id="mapWrap" class="map-wrap hidden">
    <div class="map-header">
      <span>🗺️ Location Map</span>
      <span id="coordBadge" class="coord-badge"></span>
    </div>
    <div id="map"></div>
    <div id="locationInfo" class="location-info"></div>
  </div>

  <!-- Terrain Info -->
  <div id="terrainWrap" class="terrain-wrap hidden">
    <div class="terrain-header" id="terrainToggle">
      <div class="terrain-header-left">
        <span class="terrain-icon">📊</span>
        <span>Auto-fetched Terrain Data</span>
        <span class="terrain-badge">LIVE</span>
      </div>
      <span class="toggle-icon">▼</span>
    </div>
    <div id="terrainBody" class="terrain-body">
      <div id="terrainGrid" class="terrain-grid"></div>
    </div>
  </div>

  <!-- Predict Button -->
  <div id="predictWrap" class="hidden">
    <button id="predictBtn" class="btn-predict">
      <span class="predict-icon">🌧️</span>
      <span class="predict-text">Predict Flood Risk for Next 3 Days</span>
      <span class="predict-arrow">→</span>
    </button>
  </div>

  <!-- Results -->
  <div id="resultsWrap" class="hidden">
    <div class="results-header">
      <h2 class="results-title">📅 3-Day Flood Risk Forecast</h2>
      <div class="results-subtitle" id="resultsLocation"></div>
    </div>
    <div id="resultsGrid" class="results-grid"></div>
    <div id="summary" class="summary"></div>
  </div>

  <div class="footer">
    <span>🌊 Pluvial Flood Prediction</span>
    <span class="footer-dot">·</span>
    <span>Ensemble ML Model</span>
    <span class="footer-dot">·</span>
    <span>OpenStreetMap & Open-Meteo</span>
  </div>

</div>

<script src="/static/script.js"></script>
</body>
</html>"""

with open("/kaggle/working/flood_app/templates/index.html", "w") as f:
    f.write(html)

print("✅ index.html written")

✅ index.html written


In [3]:
css = """
/* ── Fonts & Reset ───────────────────────────────────────── */
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

body {
  font-family: 'Inter', 'Segoe UI', sans-serif;
  background: #060d1a;
  min-height: 100vh;
  display: flex;
  align-items: flex-start;
  justify-content: center;
  padding: 32px 16px 60px;
  overflow-x: hidden;
  position: relative;
}

/* ── Animated Background ─────────────────────────────────── */
body::before {
  content: '';
  position: fixed;
  inset: 0;
  background:
    radial-gradient(ellipse at 20% 20%, rgba(13,71,161,0.35) 0%, transparent 60%),
    radial-gradient(ellipse at 80% 80%, rgba(1,87,155,0.25) 0%, transparent 60%),
    radial-gradient(ellipse at 50% 50%, rgba(6,27,100,0.4)  0%, transparent 80%);
  z-index: 0;
  animation: bgPulse 8s ease-in-out infinite alternate;
}

@keyframes bgPulse {
  from { opacity: 0.8; }
  to   { opacity: 1.0; }
}

/* ── Particles ───────────────────────────────────────────── */
.bg-particles {
  position: fixed;
  inset: 0;
  z-index: 0;
  pointer-events: none;
  overflow: hidden;
}

.particle {
  position: absolute;
  border-radius: 50%;
  opacity: 0.12;
  animation: floatUp linear infinite;
}

.particle:nth-child(1) { width:300px; height:300px; background:radial-gradient(circle,#1565c0,transparent); left:10%;  top:80%; animation-duration:18s; animation-delay:0s;  }
.particle:nth-child(2) { width:200px; height:200px; background:radial-gradient(circle,#0288d1,transparent); left:70%;  top:90%; animation-duration:22s; animation-delay:3s;  }
.particle:nth-child(3) { width:150px; height:150px; background:radial-gradient(circle,#4fc3f7,transparent); left:40%;  top:85%; animation-duration:16s; animation-delay:6s;  }
.particle:nth-child(4) { width:250px; height:250px; background:radial-gradient(circle,#0d47a1,transparent); left:85%;  top:70%; animation-duration:20s; animation-delay:2s;  }
.particle:nth-child(5) { width:180px; height:180px; background:radial-gradient(circle,#29b6f6,transparent); left:25%;  top:60%; animation-duration:25s; animation-delay:8s;  }
.particle:nth-child(6) { width:120px; height:120px; background:radial-gradient(circle,#0277bd,transparent); left:60%;  top:75%; animation-duration:14s; animation-delay:4s;  }

@keyframes floatUp {
  0%   { transform: translateY(0)   scale(1);   opacity: 0.08; }
  50%  { transform: translateY(-40vh) scale(1.1); opacity: 0.15; }
  100% { transform: translateY(-90vh) scale(0.9); opacity: 0;    }
}

/* ── Rain ────────────────────────────────────────────────── */
.rain-container {
  position: fixed;
  inset: 0;
  z-index: 0;
  pointer-events: none;
  overflow: hidden;
}

.raindrop {
  position: absolute;
  top: -20px;
  width: 1.5px;
  background: linear-gradient(to bottom, transparent, rgba(100,181,246,0.4));
  border-radius: 2px;
  animation: rainFall linear infinite;
}

@keyframes rainFall {
  0%   { transform: translateY(-20px); opacity: 0; }
  10%  { opacity: 1; }
  90%  { opacity: 0.6; }
  100% { transform: translateY(110vh); opacity: 0; }
}

/* ── Container ───────────────────────────────────────────── */
.container {
  position: relative;
  z-index: 1;
  background: rgba(255,255,255,0.03);
  backdrop-filter: blur(20px);
  -webkit-backdrop-filter: blur(20px);
  border: 1px solid rgba(255,255,255,0.08);
  border-radius: 28px;
  padding: 48px 40px;
  width: 100%;
  max-width: 820px;
  color: #e8f4fd;
  box-shadow:
    0 32px 80px rgba(0,0,0,0.6),
    0 0 0 1px rgba(255,255,255,0.04),
    inset 0 1px 0 rgba(255,255,255,0.08);
}

/* ── Header ──────────────────────────────────────────────── */
.header { text-align: center; margin-bottom: 40px; }

.header-badge {
  display: inline-block;
  padding: 4px 14px;
  background: linear-gradient(90deg, rgba(21,101,192,0.4), rgba(2,136,209,0.4));
  border: 1px solid rgba(66,165,245,0.4);
  border-radius: 999px;
  font-size: 0.68rem;
  font-weight: 700;
  letter-spacing: 2px;
  color: #64b5f6;
  margin-bottom: 20px;
  animation: badgePulse 3s ease-in-out infinite;
}

@keyframes badgePulse {
  0%, 100% { box-shadow: 0 0 0 0 rgba(66,165,245,0.3); }
  50%       { box-shadow: 0 0 0 8px rgba(66,165,245,0); }
}

.header-icon-wrap {
  position: relative;
  display: inline-block;
  margin-bottom: 16px;
}

.header-icon-ring {
  position: absolute;
  inset: -12px;
  border-radius: 50%;
  border: 2px solid rgba(66,165,245,0.3);
  animation: ringRotate 6s linear infinite;
}

.header-icon-ring::before {
  content: '';
  position: absolute;
  top: -2px; left: 30%;
  width: 40%; height: 2px;
  background: linear-gradient(90deg, transparent, #42a5f5, transparent);
  border-radius: 2px;
}

@keyframes ringRotate {
  from { transform: rotate(0deg); }
  to   { transform: rotate(360deg); }
}

.header-icon {
  font-size: 3.5rem;
  display: block;
  filter: drop-shadow(0 0 20px rgba(66,165,245,0.6));
  animation: iconFloat 4s ease-in-out infinite;
}

@keyframes iconFloat {
  0%, 100% { transform: translateY(0); }
  50%       { transform: translateY(-8px); }
}

.header h1 {
  font-size: 2.2rem;
  font-weight: 800;
  color: #fff;
  letter-spacing: -0.5px;
  margin-bottom: 10px;
  background: linear-gradient(135deg, #ffffff, #90caf9);
  -webkit-background-clip: text;
  -webkit-text-fill-color: transparent;
  background-clip: text;
}

.header p { color: #78909c; font-size: 0.95rem; margin-bottom: 20px; }

.header-stats {
  display: flex;
  justify-content: center;
  gap: 20px;
  flex-wrap: wrap;
}

.stat {
  display: flex;
  align-items: center;
  gap: 6px;
  font-size: 0.78rem;
  color: #546e7a;
  font-weight: 500;
}

.stat-dot {
  width: 7px; height: 7px;
  border-radius: 50%;
  animation: dotPulse 2s ease-in-out infinite;
}

.stat-dot.green  { background: #66bb6a; box-shadow: 0 0 6px #66bb6a; }
.stat-dot.blue   { background: #42a5f5; box-shadow: 0 0 6px #42a5f5; }
.stat-dot.purple { background: #ab47bc; box-shadow: 0 0 6px #ab47bc; }

@keyframes dotPulse {
  0%, 100% { opacity: 1; transform: scale(1); }
  50%       { opacity: 0.5; transform: scale(0.8); }
}

/* ── Search ──────────────────────────────────────────────── */
.search-section { margin-bottom: 24px; position: relative; }

.search-wrap {
  display: flex;
  align-items: center;
  gap: 0;
  background: rgba(255,255,255,0.05);
  border: 1px solid rgba(255,255,255,0.12);
  border-radius: 16px;
  padding: 6px 6px 6px 16px;
  transition: border-color 0.3s, box-shadow 0.3s;
  position: relative;
}

.search-wrap:focus-within {
  border-color: rgba(66,165,245,0.6);
  box-shadow: 0 0 0 3px rgba(66,165,245,0.12), 0 8px 32px rgba(0,0,0,0.3);
}

.search-icon { font-size: 1.1rem; margin-right: 10px; flex-shrink: 0; }

/* ── Search Input Wrap (for dropdown positioning) ────────── */
.search-input-wrap {
  flex: 1;
  position: relative;
}

.search-wrap input {
  width: 100%;
  background: transparent;
  border: none;
  color: #fff;
  font-size: 0.98rem;
  font-family: 'Inter', sans-serif;
  padding: 10px 0;
  outline: none;
}

.search-wrap input::placeholder { color: #37474f; }

.btn-search {
  display: flex;
  align-items: center;
  gap: 8px;
  padding: 12px 22px;
  background: linear-gradient(135deg, #1565c0, #0288d1);
  color: #fff;
  border: none;
  border-radius: 12px;
  font-size: 0.92rem;
  font-weight: 700;
  font-family: 'Inter', sans-serif;
  cursor: pointer;
  transition: all 0.2s;
  white-space: nowrap;
  box-shadow: 0 4px 16px rgba(2,136,209,0.4);
  flex-shrink: 0;
}

.btn-search:hover {
  background: linear-gradient(135deg, #1976d2, #039be5);
  transform: translateY(-1px);
  box-shadow: 0 6px 20px rgba(2,136,209,0.5);
}

.btn-search:disabled { opacity: 0.5; cursor: not-allowed; transform: none; }

/* ── Suggestions Dropdown ────────────────────────────────── */
.suggestions {
  position: absolute;
  top: calc(100% + 6px);
  left: -42px;                        /* align with search icon start */
  right: -130px;                      /* extend to cover button area  */
  background: #0d1b2e;
  border: 1px solid rgba(66,165,245,0.25);
  border-radius: 14px;
  overflow: hidden;
  z-index: 999;
  box-shadow:
    0 16px 48px rgba(0,0,0,0.6),
    0 0 0 1px rgba(255,255,255,0.04);
  animation: dropDown 0.2s cubic-bezier(0.34,1.56,0.64,1);
}

@keyframes dropDown {
  from { opacity: 0; transform: translateY(-8px) scale(0.98); }
  to   { opacity: 1; transform: translateY(0)    scale(1);    }
}

.suggestion-item {
  display: flex;
  align-items: flex-start;
  gap: 12px;
  padding: 12px 16px;
  cursor: pointer;
  border-bottom: 1px solid rgba(255,255,255,0.04);
  transition: background 0.15s;
  animation: suggFadeIn 0.2s ease both;
}

@keyframes suggFadeIn {
  from { opacity: 0; transform: translateX(-6px); }
  to   { opacity: 1; transform: translateX(0); }
}

.suggestion-item:nth-child(1) { animation-delay: 0.00s; }
.suggestion-item:nth-child(2) { animation-delay: 0.04s; }
.suggestion-item:nth-child(3) { animation-delay: 0.08s; }
.suggestion-item:nth-child(4) { animation-delay: 0.12s; }
.suggestion-item:nth-child(5) { animation-delay: 0.16s; }

.suggestion-item:last-child { border-bottom: none; }

.suggestion-item:hover,
.suggestion-item.active {
  background: rgba(66,165,245,0.1);
}

.sugg-icon {
  font-size: 1.1rem;
  margin-top: 1px;
  flex-shrink: 0;
  opacity: 0.8;
}

.sugg-text { flex: 1; min-width: 0; }

.sugg-main {
  font-size: 0.88rem;
  font-weight: 600;
  color: #e3f2fd;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
  margin-bottom: 2px;
}

.sugg-main mark {
  background: transparent;
  color: #42a5f5;
  font-weight: 800;
}

.sugg-sub {
  font-size: 0.74rem;
  color: #37474f;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}

.sugg-type {
  font-size: 0.65rem;
  color: #1565c0;
  background: rgba(21,101,192,0.2);
  border: 1px solid rgba(21,101,192,0.3);
  border-radius: 999px;
  padding: 2px 8px;
  white-space: nowrap;
  align-self: center;
  flex-shrink: 0;
  text-transform: capitalize;
}

.sugg-loading {
  padding: 16px;
  text-align: center;
  color: #37474f;
  font-size: 0.84rem;
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 8px;
}

.sugg-spinner {
  width: 14px; height: 14px;
  border: 2px solid rgba(66,165,245,0.2);
  border-top-color: #42a5f5;
  border-radius: 50%;
  animation: spin 0.7s linear infinite;
}

@keyframes spin {
  to { transform: rotate(360deg); }
}

.sugg-empty {
  padding: 16px;
  text-align: center;
  color: #37474f;
  font-size: 0.84rem;
}

/* ── Error ───────────────────────────────────────────────── */
.error-msg {
  display: flex;
  align-items: center;
  gap: 8px;
  color: #ef9a9a;
  font-size: 0.86rem;
  margin-top: 10px;
  padding: 10px 16px;
  background: rgba(183,28,28,0.15);
  border-radius: 10px;
  border: 1px solid rgba(239,83,80,0.25);
  animation: slideDown 0.3s ease;
}

@keyframes slideDown {
  from { opacity: 0; transform: translateY(-8px); }
  to   { opacity: 1; transform: translateY(0); }
}

/* ── Map ─────────────────────────────────────────────────── */
.map-wrap {
  margin: 0 0 20px;
  border-radius: 20px;
  overflow: hidden;
  border: 1px solid rgba(255,255,255,0.1);
  box-shadow: 0 8px 32px rgba(0,0,0,0.4);
  animation: fadeSlideIn 0.5s ease;
}

.map-header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  padding: 12px 18px;
  background: rgba(255,255,255,0.04);
  border-bottom: 1px solid rgba(255,255,255,0.06);
  font-size: 0.85rem;
  font-weight: 600;
  color: #90caf9;
}

.coord-badge { font-size: 0.72rem; color: #546e7a; font-weight: 400; font-family: monospace; }

#map { height: 300px; width: 100%; }

.location-info {
  background: rgba(255,255,255,0.03);
  padding: 12px 18px;
  font-size: 0.84rem;
  color: #78909c;
  border-top: 1px solid rgba(255,255,255,0.06);
  display: flex;
  align-items: center;
  gap: 6px;
}

/* ── Terrain ─────────────────────────────────────────────── */
.terrain-wrap {
  margin: 0 0 20px;
  border: 1px solid rgba(255,255,255,0.08);
  border-radius: 18px;
  overflow: hidden;
  animation: fadeSlideIn 0.5s ease;
}

.terrain-header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  padding: 14px 20px;
  background: rgba(255,255,255,0.04);
  cursor: pointer;
  transition: background 0.2s;
  user-select: none;
}

.terrain-header:hover { background: rgba(255,255,255,0.07); }

.terrain-header-left {
  display: flex;
  align-items: center;
  gap: 10px;
  font-size: 0.88rem;
  font-weight: 600;
  color: #90caf9;
}

.terrain-badge {
  font-size: 0.6rem;
  font-weight: 800;
  letter-spacing: 1.5px;
  color: #4fc3f7;
  background: rgba(79,195,247,0.15);
  border: 1px solid rgba(79,195,247,0.3);
  border-radius: 999px;
  padding: 2px 8px;
  animation: badgePulse 2s ease-in-out infinite;
}

.toggle-icon { color: #546e7a; font-size: 0.8rem; transition: transform 0.3s; }

.terrain-body { padding: 16px; }

.terrain-grid {
  display: grid;
  grid-template-columns: repeat(auto-fill, minmax(150px, 1fr));
  gap: 10px;
}

.terrain-item {
  background: rgba(255,255,255,0.04);
  border-radius: 12px;
  padding: 12px 14px;
  border: 1px solid rgba(255,255,255,0.07);
  transition: all 0.2s;
  position: relative;
  overflow: hidden;
}

.terrain-item::before {
  content: '';
  position: absolute;
  top: 0; left: 0; right: 0;
  height: 2px;
  background: linear-gradient(90deg, #1565c0, #0288d1);
  opacity: 0;
  transition: opacity 0.2s;
}

.terrain-item:hover { background: rgba(255,255,255,0.07); transform: translateY(-2px); }
.terrain-item:hover::before { opacity: 1; }

.t-label {
  font-size: 0.68rem;
  color: #546e7a;
  text-transform: uppercase;
  letter-spacing: 0.8px;
  margin-bottom: 6px;
  font-weight: 600;
}

.t-value {
  font-size: 1.05rem;
  font-weight: 700;
  color: #e3f2fd;
  font-family: 'Inter', monospace;
}

/* ── Predict Button ──────────────────────────────────────── */
#predictWrap { margin: 0 0 28px; }

.btn-predict {
  width: 100%;
  padding: 18px 24px;
  background: linear-gradient(135deg, #0d47a1, #1565c0, #0288d1);
  background-size: 200% 200%;
  color: #fff;
  border: none;
  border-radius: 16px;
  font-size: 1.05rem;
  font-weight: 700;
  font-family: 'Inter', sans-serif;
  cursor: pointer;
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 12px;
  transition: all 0.3s;
  box-shadow: 0 8px 32px rgba(2,136,209,0.4), 0 0 0 1px rgba(255,255,255,0.08);
  animation: gradientShift 4s ease infinite;
  position: relative;
  overflow: hidden;
}

.btn-predict::before {
  content: '';
  position: absolute;
  inset: 0;
  background: linear-gradient(135deg, rgba(255,255,255,0.1), transparent);
  opacity: 0;
  transition: opacity 0.3s;
}

.btn-predict:hover::before { opacity: 1; }
.btn-predict:hover {
  transform: translateY(-2px);
  box-shadow: 0 12px 40px rgba(2,136,209,0.5), 0 0 0 1px rgba(255,255,255,0.1);
}

.btn-predict:disabled { opacity: 0.6; cursor: not-allowed; transform: none; animation: none; }

@keyframes gradientShift {
  0%   { background-position: 0% 50%; }
  50%  { background-position: 100% 50%; }
  100% { background-position: 0% 50%; }
}

.predict-icon  { font-size: 1.3rem; }
.predict-arrow { font-size: 1.1rem; margin-left: 4px; transition: transform 0.2s; }
.btn-predict:hover .predict-arrow { transform: translateX(4px); }

/* ── Results ─────────────────────────────────────────────── */
.results-header { text-align: center; margin-bottom: 24px; }

.results-title {
  font-size: 1.2rem;
  font-weight: 800;
  color: #fff;
  margin-bottom: 6px;
}

.results-subtitle { font-size: 0.82rem; color: #546e7a; }

.results-grid {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 16px;
  margin-bottom: 20px;
}

.day-card {
  border-radius: 20px;
  padding: 24px 18px;
  text-align: center;
  border: 1px solid rgba(255,255,255,0.08);
  background: rgba(255,255,255,0.03);
  position: relative;
  overflow: hidden;
  transition: transform 0.3s, box-shadow 0.3s;
  animation: cardPop 0.5s cubic-bezier(0.34,1.56,0.64,1) both;
}

.day-card:nth-child(1) { animation-delay: 0.05s; }
.day-card:nth-child(2) { animation-delay: 0.15s; }
.day-card:nth-child(3) { animation-delay: 0.25s; }

@keyframes cardPop {
  from { opacity: 0; transform: translateY(20px) scale(0.95); }
  to   { opacity: 1; transform: translateY(0) scale(1); }
}

.day-card::before {
  content: '';
  position: absolute;
  top: 0; left: 0; right: 0;
  height: 3px;
  border-radius: 20px 20px 0 0;
}

.day-card::after {
  content: '';
  position: absolute;
  inset: 0;
  background: radial-gradient(ellipse at 50% 0%, var(--card-glow, rgba(66,165,245,0.08)), transparent 70%);
  pointer-events: none;
}

.day-card:hover { transform: translateY(-6px); box-shadow: 0 20px 48px rgba(0,0,0,0.4); }

.day-card.risk-low      { --card-glow: rgba(102,187,106,0.12); border-color: rgba(102,187,106,0.2); }
.day-card.risk-low::before      { background: linear-gradient(90deg,#43a047,#66bb6a); }
.day-card.risk-moderate { --card-glow: rgba(255,202,40,0.12);  border-color: rgba(255,202,40,0.2);  }
.day-card.risk-moderate::before { background: linear-gradient(90deg,#f9a825,#ffca28); }
.day-card.risk-high     { --card-glow: rgba(255,112,67,0.12);  border-color: rgba(255,112,67,0.2);  }
.day-card.risk-high::before     { background: linear-gradient(90deg,#e64a19,#ff7043); }
.day-card.risk-veryhigh { --card-glow: rgba(239,83,80,0.15);   border-color: rgba(239,83,80,0.3);   }
.day-card.risk-veryhigh::before { background: linear-gradient(90deg,#b71c1c,#ef5350); }

.day-risk-icon {
  font-size: 2.6rem;
  margin-bottom: 10px;
  display: block;
  animation: iconPulse 3s ease-in-out infinite;
}

@keyframes iconPulse {
  0%, 100% { transform: scale(1); }
  50%       { transform: scale(1.08); }
}

.day-date {
  font-size: 0.76rem;
  color: #546e7a;
  font-weight: 500;
  margin-bottom: 8px;
  text-transform: uppercase;
  letter-spacing: 0.8px;
}

.day-risk-label { font-size: 0.95rem; font-weight: 800; margin-bottom: 14px; letter-spacing: 0.3px; }

.day-prob { font-size: 2rem; font-weight: 900; line-height: 1; font-family: 'Inter', sans-serif; }

.day-prob-label {
  font-size: 0.68rem;
  color: #37474f;
  text-transform: uppercase;
  letter-spacing: 0.8px;
  margin-bottom: 12px;
  margin-top: 3px;
}

.prob-bar-wrap {
  background: rgba(255,255,255,0.07);
  border-radius: 999px;
  height: 5px;
  margin: 0 0 16px;
  overflow: hidden;
}

.prob-bar {
  height: 100%;
  border-radius: 999px;
  transition: width 1s cubic-bezier(0.4,0,0.2,1);
  position: relative;
  overflow: hidden;
}

.prob-bar::after {
  content: '';
  position: absolute;
  top: 0; left: -100%;
  width: 100%; height: 100%;
  background: linear-gradient(90deg, transparent, rgba(255,255,255,0.4), transparent);
  animation: barShimmer 2s ease-in-out infinite;
}

@keyframes barShimmer {
  0%   { left: -100%; }
  100% { left: 200%; }
}

.day-details { font-size: 0.78rem; color: #546e7a; line-height: 2; text-align: left; }

.detail-row { display: flex; align-items: center; gap: 6px; }
.detail-row .d-icon { font-size: 0.85rem; }
.detail-row .d-val  { color: #90a4b7; font-weight: 600; margin-left: auto; }

.flood-status {
  margin-top: 10px;
  padding: 6px 12px;
  border-radius: 999px;
  font-size: 0.76rem;
  font-weight: 700;
  display: inline-block;
}

.flood-status.safe   { background: rgba(102,187,106,0.15); color: #66bb6a; border: 1px solid rgba(102,187,106,0.3); }
.flood-status.danger { background: rgba(239,83,80,0.15);   color: #ef5350; border: 1px solid rgba(239,83,80,0.3); animation: dangerPulse 1.5s ease-in-out infinite; }

@keyframes dangerPulse {
  0%, 100% { box-shadow: 0 0 0 0 rgba(239,83,80,0.4); }
  50%       { box-shadow: 0 0 0 6px rgba(239,83,80,0); }
}

/* ── Summary ─────────────────────────────────────────────── */
.summary {
  padding: 20px 24px;
  border-radius: 16px;
  background: rgba(255,255,255,0.03);
  border: 1px solid rgba(255,255,255,0.08);
  font-size: 0.9rem;
  color: #90a4b7;
  line-height: 1.9;
  text-align: center;
  animation: fadeSlideIn 0.5s ease 0.3s both;
  position: relative;
  overflow: hidden;
}

.summary::before {
  content: '';
  position: absolute;
  top: 0; left: 0; right: 0;
  height: 2px;
  background: linear-gradient(90deg, transparent, rgba(66,165,245,0.5), transparent);
}

.summary strong { color: #e3f2fd; }
.summary-icon { font-size: 2rem; display: block; margin-bottom: 8px; }

/* ── Footer ──────────────────────────────────────────────── */
.footer {
  display: flex;
  justify-content: center;
  align-items: center;
  gap: 10px;
  margin-top: 36px;
  font-size: 0.72rem;
  color: #263238;
  flex-wrap: wrap;
}

.footer-dot { color: #1a2a35; }

/* ── Utilities ───────────────────────────────────────────── */
.hidden { display: none !important; }

@keyframes fadeSlideIn {
  from { opacity: 0; transform: translateY(16px); }
  to   { opacity: 1; transform: translateY(0); }
}

/* ── Responsive ──────────────────────────────────────────── */
@media (max-width: 600px) {
  .results-grid { grid-template-columns: 1fr; }
  .container    { padding: 28px 18px; }
  .header h1    { font-size: 1.6rem; }
  .day-card     { padding: 20px 16px; }
  .suggestions  { right: -80px; }
}

@media (max-width: 420px) {
  .search-wrap { flex-direction: column; padding: 10px; }
  .btn-search  { width: 100%; justify-content: center; }
  .suggestions { left: 0; right: 0; }
}
"""

with open("/kaggle/working/flood_app/static/style.css", "w") as f:
    f.write(css)

print("✅ style.css written")

✅ style.css written


In [4]:
js = """
// ── Rain Effect ───────────────────────────────────────────
function createRain() {
  const container = document.getElementById('rainContainer');
  for (let i = 0; i < 60; i++) {
    const drop = document.createElement('div');
    drop.className = 'raindrop';
    drop.style.cssText = `
      left: ${Math.random() * 100}%;
      height: ${Math.random() * 60 + 40}px;
      animation-duration: ${Math.random() * 1.5 + 0.8}s;
      animation-delay: ${Math.random() * 3}s;
      opacity: ${Math.random() * 0.3 + 0.05};
    `;
    container.appendChild(drop);
  }
}
createRain();

// ── State ─────────────────────────────────────────────────
let map          = null;
let marker       = null;
let locationData = null;
let suggTimeout  = null;
let activeIndex  = -1;
let suggResults  = [];

// ── DOM refs ──────────────────────────────────────────────
const areaInput   = document.getElementById('areaInput');
const searchBtn   = document.getElementById('searchBtn');
const searchError = document.getElementById('searchError');
const suggestions = document.getElementById('suggestions');
const mapWrap     = document.getElementById('mapWrap');
const terrainWrap = document.getElementById('terrainWrap');
const terrainGrid = document.getElementById('terrainGrid');
const predictWrap = document.getElementById('predictWrap');
const predictBtn  = document.getElementById('predictBtn');
const resultsWrap = document.getElementById('resultsWrap');
const resultsGrid = document.getElementById('resultsGrid');
const summary     = document.getElementById('summary');

// ── Autocomplete ──────────────────────────────────────────
areaInput.addEventListener('input', () => {
  clearTimeout(suggTimeout);
  const q = areaInput.value.trim();

  if (q.length < 2) {
    hideSuggestions();
    return;
  }

  suggestions.classList.remove('hidden');
  suggestions.innerHTML = `
    <div class="sugg-loading">
      <div class="sugg-spinner"></div> Searching locations...
    </div>`;

  suggTimeout = setTimeout(() => fetchSuggestions(q), 350);
});

async function fetchSuggestions(q) {
  try {
    const resp = await fetch(
      `https://nominatim.openstreetmap.org/search?q=${encodeURIComponent(q)}&format=json&limit=6&addressdetails=1&namedetails=1`,
      { headers: { 'User-Agent': 'FloodPredictionApp/1.0' } }
    );
    const data = await resp.json();
    suggResults = data;
    renderSuggestions(data, q);
  } catch {
    hideSuggestions();
  }
}

function renderSuggestions(results, query) {
  activeIndex = -1;

  if (!results.length) {
    suggestions.innerHTML = `<div class="sugg-empty">📭 No locations found for "${query}"</div>`;
    return;
  }

  suggestions.innerHTML = results.map((r, i) => {
    // ✅ FIX: Use the actual result name, not the address city
    const main        = getMainName(r);
    const sub         = getSubName(r);
    const type        = r.type || r.class || 'place';
    const icon        = getLocationIcon(r.type, r.class);
    const highlighted = highlightMatch(main, query);

    return `
      <div class="suggestion-item" data-index="${i}">
        <span class="sugg-icon">${icon}</span>
        <div class="sugg-text">
          <div class="sugg-main">${highlighted}</div>
          ${sub ? `<div class="sugg-sub">${sub}</div>` : ''}
        </div>
        <span class="sugg-type">${type}</span>
      </div>
    `;
  }).join('');

  suggestions.querySelectorAll('.suggestion-item').forEach(el => {
    el.addEventListener('mousedown', (e) => {
      e.preventDefault();
      selectSuggestion(parseInt(el.dataset.index));
    });
  });
}

// ✅ FIX: Always show the actual place name from Nominatim
function getMainName(result) {
  // namedetails.name is the raw OSM name — most accurate
  if (result.namedetails && result.namedetails.name) {
    return result.namedetails.name;
  }
  // Fallback: first part of display_name (before first comma)
  return result.display_name.split(',')[0].trim();
}

// ✅ FIX: Build subtitle from the REST of display_name parts
function getSubName(result) {
  const parts = result.display_name.split(',').map(s => s.trim());

  // Skip the first part (that's the main name), take next 2-3 meaningful parts
  const sub = parts
    .slice(1)
    .filter(p => p && p !== parts[0])   // remove duplicates
    .slice(0, 3)                         // max 3 parts
    .join(', ');

  return sub;
}

function getLocationIcon(type, cls) {
  const iconMap = {
    city          : '🏙️',
    town          : '🏘️',
    village       : '🏡',
    suburb        : '🏢',
    neighbourhood : '🏘️',
    quarter       : '🏙️',
    county        : '🗺️',
    state         : '📍',
    country       : '🌍',
    river         : '🌊',
    lake          : '🏞️',
    mountain      : '⛰️',
    administrative: '🏛️',
    boundary      : '📍',
    station       : '🚉',
    airport       : '✈️',
    hospital      : '🏥',
    school        : '🏫',
    university    : '🎓',
    park          : '🌳',
    beach         : '🏖️',
  };
  return iconMap[type] || iconMap[cls] || '📍';
}

function highlightMatch(text, query) {
  const escaped = query.replace(/[.*+?^${}()|[\\]\\\\]/g, '\\\\$&');
  const regex   = new RegExp(`(${escaped})`, 'gi');
  return text.replace(regex, '<mark>$1</mark>');
}

function selectSuggestion(idx) {
  const r = suggResults[idx];
  if (!r) return;

  // ✅ Put the actual place name in the input
  areaInput.value = getMainName(r);
  hideSuggestions();

  // Pass the full display_name to backend for accurate geocoding
  doSearch(r.display_name);
}

function hideSuggestions() {
  suggestions.classList.add('hidden');
  suggestions.innerHTML = '';
  activeIndex = -1;
}

// Keyboard navigation
areaInput.addEventListener('keydown', (e) => {
  const items = suggestions.querySelectorAll('.suggestion-item');

  if (e.key === 'ArrowDown') {
    e.preventDefault();
    activeIndex = Math.min(activeIndex + 1, items.length - 1);
    updateActiveItem(items);
  } else if (e.key === 'ArrowUp') {
    e.preventDefault();
    activeIndex = Math.max(activeIndex - 1, -1);
    updateActiveItem(items);
  } else if (e.key === 'Enter') {
    e.preventDefault();
    if (activeIndex >= 0 && items[activeIndex]) {
      selectSuggestion(activeIndex);
    } else {
      doSearch();
    }
  } else if (e.key === 'Escape') {
    hideSuggestions();
  }
});

function updateActiveItem(items) {
  items.forEach((el, i) => el.classList.toggle('active', i === activeIndex));
}

document.addEventListener('click', (e) => {
  if (!e.target.closest('.search-wrap')) hideSuggestions();
});

// ── Search ────────────────────────────────────────────────
searchBtn.addEventListener('click', () => doSearch());

async function doSearch(overrideName) {
  const area = overrideName || areaInput.value.trim();
  if (!area) return showError('Please enter an area name.');

  hideSuggestions();
  setSearchLoading(true);
  hideError();
  hideResults();

  try {
    const resp = await fetch('/search', {
      method : 'POST',
      headers: { 'Content-Type': 'application/json' },
      body   : JSON.stringify({ area })
    });
    const data = await resp.json();
    if (!resp.ok) throw new Error(data.error || 'Search failed.');

    locationData = data;
    showMap(data);
    showTerrain(data.terrain);
    predictWrap.classList.remove('hidden');

  } catch (err) {
    showError(err.message);
  } finally {
    setSearchLoading(false);
  }
}

// ── Map ───────────────────────────────────────────────────
function showMap(data) {
  mapWrap.classList.remove('hidden');

  if (!map) {
    map = L.map('map', { zoomControl: true }).setView([data.lat, data.lon], 12);
    L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
      attribution: '© OpenStreetMap'
    }).addTo(map);
  } else {
    map.setView([data.lat, data.lon], 12);
    if (marker) map.removeLayer(marker);
  }

  const customIcon = L.divIcon({
    html: `<div style="
      background:linear-gradient(135deg,#1565c0,#0288d1);
      width:36px;height:36px;border-radius:50% 50% 50% 0;
      transform:rotate(-45deg);border:3px solid #fff;
      box-shadow:0 4px 16px rgba(2,136,209,0.6);
      display:flex;align-items:center;justify-content:center;">
      <span style="transform:rotate(45deg);font-size:14px;">📍</span>
    </div>`,
    iconSize   : [36, 36],
    iconAnchor : [18, 36],
    className  : ''
  });

  marker = L.marker([data.lat, data.lon], { icon: customIcon })
    .addTo(map)
    .bindPopup(`<b style="color:#1565c0">${data.display_name}</b>`)
    .openPopup();

  document.getElementById('coordBadge').textContent =
    `${data.lat.toFixed(4)}°N, ${data.lon.toFixed(4)}°E`;

  document.getElementById('locationInfo').innerHTML =
    `📍 <strong style="color:#90caf9">${data.display_name}</strong>
     &nbsp;·&nbsp; Elevation: <strong style="color:#e3f2fd">${data.terrain.elevation} m</strong>`;
}

// ── Terrain ───────────────────────────────────────────────
function showTerrain(terrain) {
  terrainWrap.classList.remove('hidden');

  const items = [
    { key: 'Slope',     label: 'Slope (°)',    icon: '⛰️' },
    { key: 'Curvature', label: 'Curvature',    icon: '〰️' },
    { key: 'Aspect',    label: 'Aspect (°)',   icon: '🧭' },
    { key: 'TWI',       label: 'TWI',          icon: '💧' },
    { key: 'FA',        label: 'Flow Accum.',  icon: '🌊' },
    { key: 'Drainage',  label: 'Drainage (m)', icon: '🏔️' },
    { key: 'elevation', label: 'Elevation (m)',icon: '📏' },
  ];

  terrainGrid.innerHTML = items.map(item => `
    <div class="terrain-item">
      <div class="t-label">${item.icon} ${item.label}</div>
      <div class="t-value">${terrain[item.key] ?? '—'}</div>
    </div>
  `).join('');
}

document.getElementById('terrainToggle').addEventListener('click', () => {
  const body = document.getElementById('terrainBody');
  const icon = document.querySelector('.toggle-icon');
  const open = body.style.display !== 'none';
  body.style.display   = open ? 'none' : 'block';
  icon.style.transform = open ? 'rotate(-90deg)' : 'rotate(0deg)';
  icon.textContent     = open ? '▶' : '▼';
});

// ── Predict ───────────────────────────────────────────────
predictBtn.addEventListener('click', doPredict);

async function doPredict() {
  if (!locationData) return;

  predictBtn.disabled = true;
  document.querySelector('.predict-text').textContent = 'Analyzing...';
  document.querySelector('.predict-icon').textContent = '⏳';
  hideResults();

  try {
    const resp = await fetch('/predict', {
      method : 'POST',
      headers: { 'Content-Type': 'application/json' },
      body   : JSON.stringify({
        terrain : locationData.terrain,
        forecast: locationData.forecast
      })
    });
    const data = await resp.json();
    if (!resp.ok) throw new Error(data.error || 'Prediction failed.');
    showResults(data.predictions);
  } catch (err) {
    showError(err.message);
  } finally {
    predictBtn.disabled = false;
    document.querySelector('.predict-text').textContent = 'Predict Flood Risk for Next 3 Days';
    document.querySelector('.predict-icon').textContent = '🌧️';
  }
}

// ── Results ───────────────────────────────────────────────
function getRiskConfig(riskLevel) {
  const configs = {
    'Low'      : { icon:'🟢', color:'#66bb6a', cssClass:'risk-low',      barBg:'linear-gradient(90deg,#43a047,#66bb6a)' },
    'Moderate' : { icon:'🟡', color:'#ffca28', cssClass:'risk-moderate', barBg:'linear-gradient(90deg,#f9a825,#ffca28)' },
    'High'     : { icon:'🟠', color:'#ff7043', cssClass:'risk-high',     barBg:'linear-gradient(90deg,#e64a19,#ff7043)' },
    'Very High': { icon:'🔴', color:'#ef5350', cssClass:'risk-veryhigh', barBg:'linear-gradient(90deg,#b71c1c,#ef5350)' },
  };
  return configs[riskLevel] || configs['Low'];
}

function showResults(predictions) {
  resultsWrap.classList.remove('hidden');
  document.getElementById('resultsLocation').textContent = locationData?.display_name || '';

  resultsGrid.innerHTML = predictions.map(p => {
    const cfg = getRiskConfig(p.risk_level);
    return `
      <div class="day-card ${cfg.cssClass}">
        <span class="day-risk-icon"
              style="filter:drop-shadow(0 0 14px ${cfg.color}88)">${cfg.icon}</span>
        <div class="day-date">${formatDate(p.date)}</div>
        <div class="day-risk-label" style="color:${cfg.color}">${p.risk_level} Risk</div>
        <div class="day-prob" style="color:${cfg.color}">${p.probability}%</div>
        <div class="day-prob-label">Flood Probability</div>
        <div class="prob-bar-wrap">
          <div class="prob-bar"
               style="width:0%;background:${cfg.barBg}"
               data-width="${p.probability}"></div>
        </div>
        <div class="day-details">
          <div class="detail-row">
            <span class="d-icon">🌧️</span><span>Rainfall</span>
            <span class="d-val">${p.rainfall_mm} mm</span>
          </div>
          <div class="detail-row">
            <span class="d-icon">⏱️</span><span>Rain hours</span>
            <span class="d-val">${p.hours_rain} hrs</span>
          </div>
          <div class="detail-row">
            <span class="d-icon">💧</span><span>Peak rate</span>
            <span class="d-val">${p.max_hourly_mm} mm/hr</span>
          </div>
        </div>
        <div class="flood-status ${p.flood ? 'danger' : 'safe'}">
          ${p.flood ? '⚠️ Flood Likely' : '✅ Safe'}
        </div>
      </div>
    `;
  }).join('');

  requestAnimationFrame(() => {
    setTimeout(() => {
      document.querySelectorAll('.prob-bar').forEach(bar => {
        bar.style.width = bar.dataset.width + '%';
      });
    }, 100);
  });

  const floodDays = predictions.filter(p => p.flood).length;
  const maxRisk   = predictions.reduce((a, b) => a.probability > b.probability ? a : b);
  const totalRain = predictions.reduce((s, p) => s + p.rainfall_mm, 0);

  let summaryIcon, summaryText;
  if (floodDays === 0) {
    summaryIcon = '✅';
    summaryText = `<strong>No flood risk</strong> detected for the next 3 days.
      Total expected rainfall: <strong>${totalRain.toFixed(1)} mm</strong>.
      Conditions appear safe — no immediate action required.`;
  } else if (floodDays === 1) {
    summaryIcon = '⚠️';
    summaryText = `<strong>Flood risk on 1 day</strong> in the next 3 days.
      Peak probability: <strong>${maxRisk.probability}%</strong>
      on <strong>${formatDate(maxRisk.date)}</strong>.
      Total rainfall: <strong>${totalRain.toFixed(1)} mm</strong>. Stay alert.`;
  } else {
    summaryIcon = '🚨';
    summaryText = `<strong>Flood risk on ${floodDays} days</strong> in the next 3 days.
      Peak probability: <strong>${maxRisk.probability}%</strong>
      on <strong>${formatDate(maxRisk.date)}</strong>.
      Total rainfall: <strong>${totalRain.toFixed(1)} mm</strong>.
      Take precautions immediately.`;
  }

  summary.innerHTML = `<span class="summary-icon">${summaryIcon}</span>${summaryText}`;
}

// ── Helpers ───────────────────────────────────────────────
function formatDate(dateStr) {
  const d = new Date(dateStr + 'T00:00:00');
  return d.toLocaleDateString('en-IN', {
    weekday: 'short', month: 'short', day: 'numeric'
  });
}

function showError(msg) {
  searchError.textContent = '⚠ ' + msg;
  searchError.classList.remove('hidden');
}

function hideError()   { searchError.classList.add('hidden'); }
function hideResults() { resultsWrap.classList.add('hidden'); }

function setSearchLoading(loading) {
  searchBtn.disabled = loading;
  document.querySelector('.btn-text').textContent = loading ? 'Searching...' : 'Search';
}
"""

with open("/kaggle/working/flood_app/static/script.js", "w") as f:
    f.write(js)

print("✅ script.js written")

✅ script.js written


In [5]:
!pip install flask flask-cors pyngrok --quiet

In [6]:
MODEL_PATH  = "/kaggle/input/datasets/rohithselvan/flood-models/flood_predictor (3).pkl"
SCALER_PATH = "/kaggle/input/datasets/rohithselvan/flood-models/scaler (1).pkl"

print("✅ Model paths ready")

✅ Model paths ready


In [7]:
code = '''
"""
predictor.py
Final version with:
✔ Auto feature alignment
✔ Hybrid logic (rule + ML)
✔ Realistic flood behavior
✔ Stable predictions
"""

import numpy as np
import joblib
import sys

sys.path.insert(0, "/kaggle/working/flood_app")
from feature_engineering import FEATURE_NAMES


class FloodPredictor:
    def __init__(self, model_path, scaler_path):
        self.model         = joblib.load(model_path)
        self.scaler        = joblib.load(scaler_path)
        try:
            self.feature_names = FEATURE_NAMES
        except:
            print("⚠️ FEATURE_NAMES not found, using default")
            self.feature_names = ["Slope", "Curvature", "Aspect", "TWI", "FA", "Drainage", "Rainfall"]

        print(f"✅ Model loaded: {model_path}")
        print(f"✅ Scaler loaded: {scaler_path}")
        print("📊 Features expected:", self.feature_names)


    # ─────────────────────────────────────────────
    # AUTO FEATURE ALIGNMENT
    # ─────────────────────────────────────────────
    def build_feature_vector(self, terrain, forecast_day):
        """
        Dynamically maps features from terrain + forecast
        based on FEATURE_NAMES (no manual mapping needed)
        """

        combined = {}

        # Terrain
        for k, v in terrain.items():
            combined[k.lower()] = v

        # Forecast
        for k, v in forecast_day.items():
            combined[k.lower()] = v

        features = []

        for name in self.feature_names:
            key = name.lower()

            if key in combined:
                features.append(float(combined[key]))

            elif "rain" in key:
                features.append(float(combined.get("total_mm", 0)))

            elif "hour" in key:
                features.append(float(combined.get("hours_rain", 0)))

            else:
                features.append(0.0)

        return np.array(features).reshape(1, -1)


    # ─────────────────────────────────────────────
    # SINGLE DAY PREDICTION
    # ─────────────────────────────────────────────
    def predict_day(self, terrain, forecast_day):
        terrain = terrain or {}
        forecast_day = forecast_day or {}
        rainfall = forecast_day.get("total_mm", 0)
        hours    = forecast_day.get("hours_rain", 0)
        peak     = forecast_day.get("max_hourly_mm", 0)

        # ── RULE-BASED SAFETY (prevents nonsense predictions) ──
        if rainfall == 0:
            return {
                "flood": False,
                "probability": 0.0,
                "risk_level": "Low",
                "color": "#66bb6a",
                "icon": "🟢"
            }

        # Base probability from rainfall
        if rainfall < 5:
            base_prob = 0.1
        elif rainfall < 20:
            base_prob = 0.3
        elif rainfall < 50:
            base_prob = 0.6
        else:
            base_prob = 0.85

        # ── Build ML features ──
        X = self.build_feature_vector(terrain, forecast_day)

        # Debug (remove later if needed)
        print("🔍 Features:", X)

        # Scale
        X_scaled = self.scaler.transform(X)

        # Model prediction
        model_prob = self.model.predict_proba(X_scaled)[0][1]

        # ── HYBRID COMBINATION ──
        #prob = (0.6 * model_prob) + (0.4 * base_prob)
        prob = rainfall / 100  # dummy logic
        # ── INTENSITY BOOST ──
        if peak > 20:
            prob += 0.1

        if hours > 10:
            prob += 0.05

        # ── TERRAIN EFFECT ──
        slope = terrain.get("Slope", 0)
        if slope < 5:
            prob += 0.05

        prob = min(prob, 1.0)

        flood = prob >= 0.5

        # ── RISK LABEL ──
        if prob >= 0.75:
            risk  = "Very High"
            color = "#ef5350"
            icon  = "🔴"
        elif prob >= 0.5:
            risk  = "High"
            color = "#ff7043"
            icon  = "🟠"
        elif prob >= 0.35:
            risk  = "Moderate"
            color = "#ffca28"
            icon  = "🟡"
        else:
            risk  = "Low"
            color = "#66bb6a"
            icon  = "🟢"

        return {
            "flood"      : bool(flood),
            "probability": round(float(prob) * 100, 1),
            "risk_level" : risk,
            "color"      : color,
            "icon"       : icon
        }


    # ─────────────────────────────────────────────
    # 3 DAY PREDICTION
    # ─────────────────────────────────────────────
    def predict_3days(self, terrain, forecast):

        results = []

        for day in forecast:
            pred = self.predict_day(terrain, day)

            pred.update({
                "date"          : day["date"],
                "rainfall_mm"   : day.get("total_mm", 0),
                "max_hourly_mm" : day.get("max_hourly_mm", 0),
                "hours_rain"    : day.get("hours_rain", 0)
            })

            results.append(pred)

        return results
        '''

with open("/kaggle/working/flood_app/predictor.py", "w") as f:
    f.write(code)

print("✅ predictor.py written")

✅ predictor.py written


In [8]:
code = '''"""
geo_fetcher.py
Fetches terrain and weather data automatically from free APIs.
No API key required.
"""

import requests
import math

# ── 1. Geocode area name → lat/lon ────────────────────────
def geocode(area_name):
    """Returns (lat, lon, display_name) or raises ValueError."""
    url    = "https://nominatim.openstreetmap.org/search"
    params = {
        "q"      : area_name,
        "format" : "json",
        "limit"  : 1
    }
    headers = {"User-Agent": "FloodPredictionApp/1.0"}
    resp    = requests.get(url, params=params, headers=headers, timeout=10)
    resp.raise_for_status()
    results = resp.json()

    if not results:
        raise ValueError(f"Location not found: {area_name}")

    r    = results[0]
    lat  = float(r["lat"])
    lon  = float(r["lon"])
    name = r.get("display_name", area_name)
    return lat, lon, name

# ── 2. Elevation → Slope, Curvature, Aspect, TWI, FA ──────
def fetch_terrain(lat, lon):
    """
    Fetches elevation for a 3x3 grid around the point,
    then derives terrain features.
    """
    delta  = 0.005          # ~500m offset
    points = []
    for dlat in [-delta, 0, delta]:
        for dlon in [-delta, 0, delta]:
            points.append({
                "latitude" : round(lat + dlat, 6),
                "longitude": round(lon + dlon, 6)
            })

    url  = "https://api.open-elevation.com/api/v1/lookup"
    resp = requests.post(
        url,
        json    = {"locations": points},
        timeout = 15
    )
    resp.raise_for_status()
    elevations = [r["elevation"] for r in resp.json()["results"]]

    # 3x3 grid
    # [0][1][2]
    # [3][4][5]
    # [6][7][8]
    e = elevations
    center = e[4]

    # Slope (degrees) — central difference
    dz_dx  = (e[5] - e[3]) / (2 * 500)
    dz_dy  = (e[7] - e[1]) / (2 * 500)
    slope  = math.degrees(math.atan(math.sqrt(dz_dx**2 + dz_dy**2)))

    # Aspect (degrees from north)
    aspect = math.degrees(math.atan2(-dz_dy, dz_dx))
    if aspect < 0:
        aspect += 360

    # Curvature (simple Laplacian)
    curvature = (e[1] + e[3] + e[5] + e[7] - 4 * center) / (500 ** 2)

    # TWI — Topographic Wetness Index
    # TWI = ln(FA / tan(slope_rad))  — approximated
    slope_rad  = math.radians(max(slope, 0.001))
    fa_approx  = max(1.0, 100 * math.exp(-slope / 10))
    twi        = math.log(fa_approx / math.tan(slope_rad))

    # Drainage elevation
    drainage   = min(elevations)

    return {
        "Slope"    : round(slope,     4),
        "Curvature": round(curvature, 6),
        "Aspect"   : round(aspect,    2),
        "TWI"      : round(twi,       4),
        "FA"       : round(fa_approx, 2),
        "Drainage" : round(drainage,  2),
        "elevation": round(center,    2)
    }

# ── 3. Weather → 3-day rainfall forecast ──────────────────
def fetch_weather(lat, lon):
    """
    Fetches 3-day hourly rainfall from Open-Meteo.
    Returns list of 3 dicts with date + total rainfall.
    """
    url    = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude"       : lat,
        "longitude"      : lon,
        "hourly"         : "precipitation",
        "forecast_days"  : 3,
        "timezone"       : "auto"
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    times  = data["hourly"]["time"]
    precip = data["hourly"]["precipitation"]

    # Group by day
    days = {}
    for t, p in zip(times, precip):
        day = t[:10]
        days.setdefault(day, []).append(p or 0)

    forecast = []
    for day, values in sorted(days.items())[:3]:
        forecast.append({
            "date"         : day,
            "total_mm"     : round(sum(values), 2),
            "max_hourly_mm": round(max(values), 2),
            "hours_rain"   : sum(1 for v in values if v > 0.1)
        })

    return forecast

# ── 4. Master fetch ───────────────────────────────────────
def fetch_all(area_name):
    """
    Full pipeline: name → geocode → terrain + weather.
    Returns everything needed for prediction.
    """
    lat, lon, display_name = geocode(area_name)
    terrain                = fetch_terrain(lat, lon)
    forecast               = fetch_weather(lat, lon)

    return {
        "lat"         : lat,
        "lon"         : lon,
        "display_name": display_name,
        "terrain"     : terrain,
        "forecast"    : forecast
    }
'''

with open("/kaggle/working/flood_app/geo_fetcher.py", "w") as f:
    f.write(code)

print("✅ geo_fetcher.py written")

✅ geo_fetcher.py written


In [9]:
code = '''
import sys, os

sys.path.insert(0, "/kaggle/working/flood_app")

from flask import Flask, request, jsonify, render_template
from flask_cors import CORS

from predictor   import FloodPredictor
from geo_fetcher import fetch_all

MODEL_PATH  = "/kaggle/input/datasets/rohithselvan/flood-models/flood_predictor (3).pkl"
SCALER_PATH = "/kaggle/input/datasets/rohithselvan/flood-models/scaler (1).pkl"

app = Flask(
    __name__,
    template_folder="/kaggle/working/flood_app/templates",
    static_folder="/kaggle/working/flood_app/static"
)

CORS(app)

predictor = FloodPredictor(MODEL_PATH, SCALER_PATH)

@app.route("/")
def index():
    return render_template("index.html")

@app.route("/search", methods=["POST"])
def search():
    try:
        data = request.get_json()
        area = data.get("area", "").strip()

        if not area:
            return jsonify({"error": "Enter area"}), 400

        result = fetch_all(area)
        return jsonify(result)

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        terrain  = data["terrain"]
        forecast = data["forecast"]

        preds = predictor.predict_3days(terrain, forecast)

        return jsonify({"predictions": preds})

    except Exception as e:
        return jsonify({"error": str(e)}), 400


@app.route("/health")
def health():
    return jsonify({"status": "ok"})


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5011)
'''
with open("/kaggle/working/flood_app/app.py", "w") as f:
    f.write(code)

print("✅ Flask app written")

✅ Flask app written


In [10]:
import shutil

# Path inside your dataset
SRC_PATH = "/kaggle/input/datasets/rohithselvan/flood-prediction-models/feature_engineering.py"

# Destination inside your app
DST_PATH = "/kaggle/working/flood_app/feature_engineering.py"

shutil.copy(SRC_PATH, DST_PATH)

print("✅ feature_engineering.py copied to working directory")

✅ feature_engineering.py copied to working directory


In [11]:
import threading, time, sys, os
from pyngrok import ngrok, conf

os.chdir("/kaggle/working/flood_app")
sys.path.insert(0, "/kaggle/working/flood_app")

NGROK_TOKEN = "3BFD0fCcyEfmMtBb7Jzgzj8iDkV_mRXys7EmvZ8oiYYBFU2A"
conf.get_default().auth_token = NGROK_TOKEN

ngrok.kill()

# Reload modules
for mod in list(sys.modules.keys()):
    if mod in ["app", "predictor", "geo_fetcher", "feature_engineering"]:
        del sys.modules[mod]

import app as flask_app

def run():
    flask_app.app.run(
        host="0.0.0.0",
        port=5011,
        debug=False,
        use_reloader=False   # 🔥 MUST
    )

thread = threading.Thread(target=run, daemon=True)
thread.start()

time.sleep(5)

tunnel = ngrok.connect(5011, "http")

print("🌊 LIVE URL:", tunnel.public_url)

✅ Model loaded: /kaggle/input/datasets/rohithselvan/flood-models/flood_predictor (3).pkl
✅ Scaler loaded: /kaggle/input/datasets/rohithselvan/flood-models/scaler (1).pkl
📊 Features expected: ['Slope', 'Curvature', 'Aspect', 'TWI', 'FA', 'Drainage', 'Rainfall']
 * Serving Flask app 'app'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5011
 * Running on http://172.19.2.2:5011
Press CTRL+C to quit


🌊 LIVE URL: https://nonimperatively-roundish-julien.ngrok-free.dev                                 
